In [138]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matrixprofile import *
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [139]:
data = pd.read_csv('TS_pulite/df_finale.csv')

In [140]:
data

,0,1,2,3,4,5,6,7,8,9,...,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Short,genre_Sport,genre_Thriller,genre_War,genre_Western,rating_encoded
0,3.203808,3.310528,3.080549,2.714583,2.409941,1.993775,1.583635,1.246871,0.796810,0.508554,...,0,0,0,1,0,0,0,0,0,4.0
1,1.641869,1.789420,1.837915,1.826346,1.841613,1.608536,1.322736,1.058922,0.766201,0.509104,...,0,0,1,0,0,0,0,0,0,4.0
2,2.146533,2.074483,1.922000,1.688202,1.367001,1.099541,0.879503,0.663925,0.466093,0.366285,...,0,0,0,0,0,0,0,0,0,4.0
3,1.244228,0.974971,0.792436,0.658828,0.110609,-0.230373,-0.290343,-0.352859,-0.421936,-0.490316,...,0,0,0,1,0,0,0,0,0,4.0
4,2.958126,2.791581,2.692438,2.631157,2.300224,2.052480,1.904554,1.740678,1.606702,1.489338,...,0,0,0,0,0,0,0,0,0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1129,3.930861,3.678304,3.339795,2.963937,2.483528,2.027932,1.564115,1.153334,0.836055,0.641848,...,0,1,0,0,0,0,1,0,0,2.0
1130,2.917085,2.810270,2.618698,2.314326,2.073896,2.056742,1.890531,1.667496,1.459727,1.321335,...,0,0,0,0,0,0,0,0,0,2.0
1131,3.558092,3.264517,2.947922,2.588963,2.087472,1.656590,1.244943,0.928361,0.700896,0.570797,...,0,0,0,0,0,0,1,0,0,4.0
1132,3.468692,3.669017,3.510363,3.077339,2.725131,2.253547,1.699255,1.092181,0.427260,0.010953,...,0,0,0,0,0,0,0,0,0,4.0


In [141]:
df_timeseries = data.iloc[:, :100]
classes = data['rating_encoded']

In [142]:
# Seleziona prime 100 colonne + ultima colonna
df= data.iloc[:, list(range(100)) + [-1]]

In [144]:
df

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,rating_encoded
0,3.203808,3.310528,3.080549,2.714583,2.409941,1.993775,1.583635,1.246871,0.796810,0.508554,...,-0.773251,-0.785931,-0.803325,-0.819292,-0.826546,-0.800523,-0.780994,-0.758548,-0.731235,4.0
1,1.641869,1.789420,1.837915,1.826346,1.841613,1.608536,1.322736,1.058922,0.766201,0.509104,...,0.091804,-0.046413,-0.296589,-0.555315,-0.811598,-0.923964,-0.900554,-0.895424,-0.931546,4.0
2,2.146533,2.074483,1.922000,1.688202,1.367001,1.099541,0.879503,0.663925,0.466093,0.366285,...,-0.758995,-0.769806,-0.782269,-0.789811,-0.797837,-0.803927,-0.801292,-0.798581,-0.795879,4.0
3,1.244228,0.974971,0.792436,0.658828,0.110609,-0.230373,-0.290343,-0.352859,-0.421936,-0.490316,...,-0.366959,-0.498499,-0.525716,-0.548996,-0.568911,-0.587338,-0.602909,-0.612658,-0.613246,4.0
4,2.958126,2.791581,2.692438,2.631157,2.300224,2.052480,1.904554,1.740678,1.606702,1.489338,...,-0.646453,-0.645161,-0.645457,-0.647903,-0.651366,-0.656044,-0.661063,-0.665579,-0.668798,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1129,3.930861,3.678304,3.339795,2.963937,2.483528,2.027932,1.564115,1.153334,0.836055,0.641848,...,-0.582350,-0.584734,-0.588441,-0.591204,-0.592680,-0.593513,-0.593650,-0.593869,-0.594236,2.0
1130,2.917085,2.810270,2.618698,2.314326,2.073896,2.056742,1.890531,1.667496,1.459727,1.321335,...,-0.688884,-0.696743,-0.702418,-0.706121,-0.709908,-0.710889,-0.710866,-0.709265,-0.704758,2.0
1131,3.558092,3.264517,2.947922,2.588963,2.087472,1.656590,1.244943,0.928361,0.700896,0.570797,...,-0.547282,-0.579174,-0.602577,-0.644120,-0.707286,-0.765331,-0.802150,-0.828583,-0.841612,4.0
1132,3.468692,3.669017,3.510363,3.077339,2.725131,2.253547,1.699255,1.092181,0.427260,0.010953,...,-0.486256,-0.508885,-0.540992,-0.570104,-0.590912,-0.601979,-0.604438,-0.608394,-0.614915,4.0


In [145]:
def extract_motifs_matrix_profile(time_series, motif_length=20, max_motifs=5):
    """
    Estrae motifs usando matrixProfile con stomp
    """
    # Calcola matrix profile usando stomp
    profile = matrixProfile.stomp(time_series, motif_length)
    
    # Estrai motifs
    mtfs, motif_d = motifs.motifs(time_series, profile, max_motifs=max_motifs)
    
    # Estrai le subsequenze motif dalla time series
    extracted_motifs = []
    
    for motif_indices in mtfs:
        if len(motif_indices) > 0:
            # Prendi il primo indice del motif
            start_idx = motif_indices[0]
            motif_array = time_series[start_idx:start_idx + motif_length]
            extracted_motifs.append(motif_array)
    
    return extracted_motifs

In [146]:
def process_time_series_dataset(df, motif_length=20, max_motifs_per_series=5):
    """
    Processa un dataset di time series per estrarre motifs usando matrixProfile
    
    Parameters:
    df: DataFrame con prime 100 colonne come timestamps e class_label dopo
    motif_length: lunghezza dei motifs da estrarre (parametro m)
    max_motifs_per_series: numero massimo di motifs per serie
    """
    
    # Separa time series data e class labels
    time_series_cols = df_timeseries  # Prime 100 colonne
    class_col = classes  # Colonna 101 (class_label)
    
    motif_dataset = []
    
    for series_id in range(len(df)):
    # Estrai la time series dalla riga series_id
        time_series = df.iloc[series_id, :100].values  # Prime 100 colonne della riga
        class_label = df.iloc[series_id, 100]  # Colonna 101 della riga
       
       
        extracted_motifs = extract_motifs_matrix_profile(time_series, motif_length, 
                                                       max_motifs_per_series)
       
        for motif_array in extracted_motifs:
            motif_dict = {
                'motif': motif_array,
                'class': class_label,
                'series_id': series_id
            }
            motif_dataset.append(motif_dict)
    
    return motif_dataset

In [147]:
motifs_dataset = process_time_series_dataset(df, motif_length=7, max_motifs_per_series=5)

In [148]:
df_motifs = pd.DataFrame(motifs_dataset)
grouped = df_motifs.groupby('class')

In [149]:
from dtaidistance import dtw

def avg_similarity(motif, motif_list):
    return np.mean([dtw.distance(motif, other) for other in motif_list])

In [154]:
top_motifs = {}
for cls, group in grouped:
    motifs = group['motif'].tolist()
    scores = [avg_similarity(m, motifs) for m in motifs]
    top_indices = np.argsort(scores)[:3]
    top_motifs[cls] = top_indices

In [155]:
top_motifs

{0.0: array([ 9,  0, 19]),
 1.0: array([429, 350, 262]),
 2.0: array([981, 787, 156]),
 3.0: array([724, 290, 195]),
 4.0: array([1076, 1297,  404])}

### Comparison with shapelets

In [159]:
from sklearn.preprocessing import MinMaxScaler

In [160]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from tslearn.shapelets import ShapeletModel, grabocka_params_to_shapelet_size_dict
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [161]:
import joblib

# Salva il modello
X_shp_train = joblib.load( "X_shp_train.pkl")
X_shp_test = joblib.load("X_shp_test.pkl")

In [162]:
shapelets = np.load("shapelets.npy")

In [164]:
X_train = pd.read_csv("TS_pulite/X_train.csv")
X_test = pd.read_csv("TS_pulite/X_test.csv")
y_train = pd.read_csv("TS_pulite/y_train.csv")
y_test = pd.read_csv("TS_pulite/y_test.csv")
X_train = X_train.iloc[:, :100]
X_test = X_test.iloc[:, :100]
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [165]:
scaler_ts = MinMaxScaler()
X_train_scaled = scaler_ts.fit_transform(X_train)
X_test_scaled = scaler_ts.transform(X_test)

In [166]:
# Calcola shapelet_sizes con la funzione grabocka_params_to_shapelet_size_dict
shapelet_sizes = grabocka_params_to_shapelet_size_dict(
    n_ts=X_train_scaled.shape[0],
    ts_sz=X_train_scaled.shape[1],
    n_classes=len(np.unique(y_train)),
    l=0.1,    # lunghezza media shapelet = 10% della serie temporale
    r=10      # shapelet per classe
)

print("Shapelet sizes dict:", shapelet_sizes)

Shapelet sizes dict: {10: 5, 20: 5, 30: 5, 40: 5, 50: 5, 60: 5, 70: 4, 80: 4, 90: 4, 100: 3}


In [167]:
shp_model = ShapeletModel(
    n_shapelets_per_size=shapelet_sizes,
    optimizer='adam',
    weight_regularizer=0.01,
    max_iter=100,
    random_state=42,
    verbose=1
)

shp_model.fit(X_train_scaled, y_train)
X_shp_train = shp_model.transform(X_train_scaled)
X_shp_test = shp_model.transform(X_test_scaled)

/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/tslearn/shapelets/shapelets.py:353: FutureWarning: The default value for 'scale' is set to False in version 0.4 to ensure backward compatibility, but is likely to change in a future version.
  warnings.warn("The default value for 'scale' is set to False "
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/laviniarotellini/Desktop/DM2/.venv/lib/python3.13/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = clos

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - categorical_accuracy: 0.2043 - categorical_crossentropy: 1.6184 - loss: 1.7007
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - categorical_accuracy: 0.2043 - categorical_crossentropy: 1.6123 - loss: 1.6923 
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - categorical_accuracy: 0.2043 - categorical_crossentropy: 1.6063 - loss: 1.6842 
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - categorical_accuracy: 0.2043 - categorical_crossentropy: 1.6004 - loss: 1.6763 
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - categorical_accuracy: 0.2068 - categorical_crossentropy: 1.5946 - loss: 1.6686 
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - categorical_accuracy: 0.2068 - categorical_crossentropy: 1.5889 - loss: 1.6610 
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - categorical_accuracy: 0.2282 - categorical_crossentropy: 1.5834 - loss: 1.6537 
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - categorical_accuracy: 0.2358 - ca

In [168]:
clf = RandomForestClassifier(criterion= 'gini', max_depth= 10, max_features= 'log2', min_samples_leaf=2, 
                             min_samples_split= 10, n_estimators= 100, random_state=42,  class_weight='balanced')
clf.fit(X_shp_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       max_features='log2', min_samples_leaf=2,
                       min_samples_split=10, random_state=42)

In [169]:
importances = clf.feature_importances_
importances

array([0.04225182, 0.02830017, 0.02206277, 0.02309936, 0.02585518,
       0.01828269, 0.02107572, 0.01851023, 0.02594325, 0.0124943 ,
       0.02964752, 0.02757773, 0.01970194, 0.03097771, 0.01553071,
       0.01394733, 0.03545198, 0.02644882, 0.02027923, 0.02545841,
       0.02131809, 0.01686727, 0.01620075, 0.01928366, 0.01926042,
       0.03272206, 0.00978771, 0.01588713, 0.03112774, 0.02000891,
       0.02789299, 0.0178707 , 0.02151675, 0.02354161, 0.02134508,
       0.01932637, 0.01245232, 0.02707381, 0.02344805, 0.01772054,
       0.02047436, 0.01137865, 0.0260099 , 0.02921873, 0.01536953])

In [170]:
indices = np.argsort(importances)[::-1]

# Stampa le prime 10 più importanti
for i in range(10):
    print(f"Shapelet {indices[i]} - Importance: {importances[indices[i]]:.4f}")

Shapelet 0 - Importance: 0.0423
Shapelet 16 - Importance: 0.0355
Shapelet 25 - Importance: 0.0327
Shapelet 28 - Importance: 0.0311
Shapelet 13 - Importance: 0.0310
Shapelet 10 - Importance: 0.0296
Shapelet 43 - Importance: 0.0292
Shapelet 1 - Importance: 0.0283
Shapelet 30 - Importance: 0.0279
Shapelet 11 - Importance: 0.0276


In [ ]:
import numpy as np

# X_shp_train.shape = (n_samples, n_shapelets)
shapelet_idx = 0
mean_dist_per_class = []
for c in np.unique(y_train):
    mean_dist_per_class.append(np.mean(X_shp_train[y_train==c, shapelet_idx]))
    
print(mean_dist_per_class)

[np.float32(0.015186913), np.float32(0.01493587), np.float32(0.015336698), np.float32(0.016029611), np.float32(0.016543243)]


In [171]:
shapelets = shp_model.shapelets_as_time_series_

In [ ]:
shapelets

array([[[0.10467423],
        [0.0287637 ],
        [0.37565529],
        ...,
        [       nan],
        [       nan],
        [       nan]],

       [[0.44120014],
        [0.45257041],
        [0.49643984],
        ...,
        [       nan],
        [       nan],
        [       nan]],

       [[0.42048326],
        [0.33613968],
        [0.76242936],
        ...,
        [       nan],
        [       nan],
        [       nan]],

       ...,

       [[0.06072078],
        [0.08534915],
        [0.0968128 ],
        ...,
        [0.70972741],
        [0.70906377],
        [0.72183454]],

       [[1.21263301],
        [1.26668441],
        [1.28460956],
        ...,
        [0.66104454],
        [0.65815914],
        [0.67196667]],

       [[0.29439926],
        [0.30583304],
        [0.31330544],
        ...,
        [0.33530098],
        [0.34501782],
        [0.36817881]]], shape=(45, 100, 1))